### Retiro Fugas

In [1]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

fecha_mes_base='2026-08-01'

filename='actualizacion_tc_20260817_2.xlsx'
name_dni='DNI'

ruta_archivo = os.path.join(ruta_diners_tc, filename)
df = pd.read_excel(ruta_archivo)

df[f"{name_dni}"] = (
    df[f"{name_dni}"]
    .astype(str)
    .str.zfill(8)
)
print(df.shape)
df = df.rename(columns={
    f'{name_dni}': 'NUMERO_DOCUMENTO'
})
df.head()
server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)



(50145, 3)


In [4]:
df.shape

(35333, 2)

In [2]:
df.columns

Index(['fuente', 'NUMERO_DOCUMENTO', 'mes'], dtype='object')

In [6]:
df['fuente'].unique()

array(['Sigma', 'Alpha', 'Beta', 'Gamma', 'Pi', 'Zeta', 'SIGMA'],
      dtype=object)

In [8]:
df=df[['NUMERO_DOCUMENTO','FUENTE']]

In [8]:
df.head()

,fuente,NUMERO_DOCUMENTO,mes
0,Sigma,46594750,junio
1,Alpha,45494166,junio
2,Alpha,70124763,junio
3,Sigma,70505237,junio
4,Sigma,41602978,junio


In [7]:

df.to_sql(
    name="cruce_dinerstc",
    con=engine_kishin,
    if_exists="append",
    index=False,
    chunksize=1000
)


15195

In [17]:
try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE a
            SET a.N_BASE = b.fuente
            from DANTALION.dbo.Base_Maestra_Diners_TC a
            inner join DANTALION.dbo.cruce_dinerstc B
            ON A.NUMERO_DOCUMENTO=B.NUMERO_DOCUMENTO
            WHERE A.fecha_envio>='2026-07-01'
            and A.fecha_envio<'2026-08-01'
            and b.mes='julio'
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)

Filas actualizadas: 36911


In [ ]:
try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE a
            SET a.N_BASE = b.fuente
            from DANTALION.dbo.Base_Maestra_Diners_TC a
            inner join DANTALION.dbo.cruce_dinerstc B
            ON A.NUMERO_DOCUMENTO=B.NUMERO_DOCUMENTO
            WHERE A.fecha_envio>='2026-08-01'
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)

In [9]:
server_sql = server_zeus
db_sql = "MAEBA"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_MAEBA = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)


In [11]:
df.to_sql(
    name="cruce_dinerstc",
    con=engine_MAEBA,
    if_exists="append",
    index=False,
    chunksize=1000
)


15195

In [14]:
try:
    with engine_MAEBA.begin() as conn:
        query = f"""
            UPDATE a
            SET a.N_BASE = b.fuente
            from MAEBA.ADM_OBJ_TG.tGestionMesDinersTc a
            inner join MAEBA.dbo.cruce_dinerstc B
            ON A.NUMERO_DOCUMENTO COLLATE Latin1_General_CI_AI=B.NUMERO_DOCUMENTO
            WHERE A.AÑO_DURACION_BASE=2026
            and  a.tMesGestion=b.mes
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)

Filas actualizadas: 50145


In [18]:
from sqlalchemy import text

with engine_kishin.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS cruce_dinerstc"))
    # conn.execute(text("TRUNCATE TABLE tb_funnel_reclutamiento"))

In [12]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Diners_tc", "SP tNumeros diners TC")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Diners_tc", "SP actualizar diners TC Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Diners_tc", "SP actualizar diners TC SA")

SP tNumeros diners TC | realizado | duración: 29.3 seg
SP actualizar diners TC Zeus | realizado | duración: 209.02 seg
SP actualizar diners TC SA | realizado | duración: 3.09 seg
